# Chapter 9. Boundary and automation audit

**Learning objective.** Retain and interpret continuous boundary differences while testing whether correction of automatic defaults changes labels, boundaries, time, or confidence.

**Estimated time:** 75–90 minutes. **Exercises:** 9.3 and 9.4. All boundaries and automatic proposals are explicitly **synthetic teaching fixtures**.

## Contract and integrity gate

The notebook compares two independent first-pass boundary series and a counterbalanced automation audit. Continuous signed and absolute differences remain in the primary output even when tolerance indicators are added. Outputs are `boundary_differences.tsv`, `boundary_diagnostics.html`, `automation_bias.tsv`, `review_decisions.tsv`, `analysis_decisions.yaml`, `run_provenance.json`, and `decision_record.json`.

In [ ]:
find_repository <- function(start = getwd()) {
  current <- normalizePath(start, mustWork = TRUE)
  repeat {
    if (file.exists(file.path(current, "companion", "data", "MANIFEST.tsv"))) return(current)
    parent <- dirname(current)
    if (identical(parent, current)) stop("Run from the repository root or a subdirectory.")
    current <- parent
  }
}
REPOSITORY_ROOT <- find_repository()
DATA_ROOT <- file.path(REPOSITORY_ROOT, "companion", "data", "ch09", "synthetic_annotation")
if (!dir.exists(DATA_ROOT)) stop("Chapter 9 fixture absent. Run companion/scripts/generate_ch09_fixture.py explicitly; no data are substituted silently.")
source(file.path(REPOSITORY_ROOT, "companion", "r", "ch09.R"))
cat("DATA STATUS: SYNTHETIC_TEACHING_FIXTURE — independent generated judgments, not human annotations.
")

In [ ]:
OUTPUT <- file.path(REPOSITORY_ROOT, "companion", "outputs", "ch09", "02_boundary_and_automation_audit")
result <- run_boundary_and_automation_audit(
  file.path(DATA_ROOT, "annotation_schema.yaml"), file.path(DATA_ROOT, "annotation_assignments.tsv"),
  file.path(DATA_ROOT, "boundary_judgments.tsv"), file.path(DATA_ROOT, "automatic_proposal_audit.tsv"), OUTPUT
)
stopifnot(nrow(result$differences) == 240)
stopifnot(all(c("signed_difference_ms", "absolute_difference_ms") %in% names(result$differences)))
print(result$summary)
print(result$bias)
print(result$review)

## Learner decision

Use the full difference distribution to decide whether the same boundaries are interchangeable for 15 ms and 100 ms duration contrasts. Then compare from-scratch and correction modes, distinguishing faster annotation from default anchoring. Recommend a counterbalanced review design and name the fields that must remain auditable.

In [ ]:
learner_decision <- list(data_status = "SYNTHETIC_TEACHING_FIXTURE", decision_status = "undecided", contrast_15ms = "Learner response required.", contrast_100ms = "Learner response required.", automation_review = "Learner response required.", warning = "Tolerance indicators do not replace the continuous difference distribution.")
jsonlite::write_json(learner_decision, file.path(OUTPUT, "decision_record.json"), pretty = TRUE, auto_unbox = TRUE)
stopifnot(all(file.exists(file.path(OUTPUT, c("boundary_differences.tsv", "boundary_diagnostics.html", "automation_bias.tsv", "review_decisions.tsv", "analysis_decisions.yaml", "run_provenance.json", "decision_record.json")))))
print(learner_decision)